In [1]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import ipywidgets as widgets

# Load CSV
import os
import re
import tkinter as tk
from tkinter import filedialog

# -----------------------------
# Settings
# -----------------------------
theoretical_max_y_by_pass = {1: 281.674, 2:282.0 }
pixel_to_mm = 0.507
output_csv_name = "offset_pass_nodes.csv"

# --------------------------
# USER FILE SELECTION
# --------------------------
root = tk.Tk()
root.withdraw()  # Hide main window
root.attributes("-topmost", True)
root.lift()
root.update()

print("Select Merged file")
csv_file = filedialog.askopenfilename(
    parent=root,
    title="Select Merged file",
    filetypes=[("CSV files", "*.csv")],
    initialdir='automation_test/data/pa6cf/cal_21_high_gain'
)
root.attributes("-topmost", False)
root.destroy()

def extract_radius_from_path(path):
    radius_pattern = re.compile(r"(?<![A-Za-z0-9])r(\d+(?:\.\d+)?)(?![A-Za-z0-9])", re.IGNORECASE)
    path_parts = os.path.normpath(path).split(os.sep)
    for part in reversed(path_parts):
        match = radius_pattern.search(os.path.splitext(part)[0])
        if match:
            return float(match.group(1))
    raise ValueError("Could not infer radius from selected path. Expected a folder or filename like R2.")

radius_value = extract_radius_from_path(csv_file)

# Load CSV and treat timestamps as elapsed milliseconds
df = pd.read_csv(csv_file)
original_columns = df.columns.tolist()
df["timestamp"] = pd.to_numeric(df["timestamp"], errors="coerce")
df = df.dropna(subset=["timestamp"]).sort_values("timestamp").copy()

# Time since start without converting to calendar dates from the CSV values
start_timestamp = df["timestamp"].iloc[0]
df["time_ms"] = (df["timestamp"] - start_timestamp).astype(float)
df["time_from_start"] = pd.to_timedelta(df["time_ms"], unit="ms")

# Anchor elapsed time to a fixed date so Plotly can format the axis as HH:MM:SS.ms
df["plot_time"] = pd.Timestamp("2000-01-01") + df["time_from_start"]

# Keep an explicit elapsed-time string for hover labels to avoid Plotly datetime resets
elapsed_ms_int = df["time_ms"].round().astype("int64")
df["elapsed_hms"] = [
    f"{ms // 3600000:02d}:{(ms % 3600000) // 60000:02d}:{(ms % 60000) // 1000:02d}.{ms % 1000:03d}"
    for ms in elapsed_ms_int
]

# ----- Layer detection -----
z_tol = 0.05
min_nodes = 10
expected_layer_height = 0.3
layer_height_tol = 0.1

def is_stable_window(values, start_idx, window_size, tol):
    if start_idx + window_size > len(values):
        return False, np.nan
    window = values[start_idx:start_idx + window_size]
    center = float(np.median(window))
    return bool(np.all(np.abs(window - center) <= tol)), center

z_values = df["Z"].to_numpy(dtype=float)
layer_ids = np.full(len(df), -1, dtype=int)
layer_z_values = np.full(len(df), np.nan, dtype=float)

current_idx = 0
layer_id = 1
previous_layer_z = None

while current_idx < len(z_values):
    stable_found = False
    while current_idx < len(z_values):
        stable_found, current_layer_z = is_stable_window(z_values, current_idx, min_nodes, z_tol)
        if not stable_found:
            current_idx += 1
            continue

        if previous_layer_z is None:
            break

        layer_step = current_layer_z - previous_layer_z
        same_layer = abs(layer_step) <= z_tol
        expected_step = abs(layer_step - expected_layer_height) <= layer_height_tol
        if same_layer or expected_step:
            break

        current_idx += 1
        stable_found = False

    if not stable_found:
        break

    in_layer_indices = []
    scan_idx = current_idx

    while scan_idx < len(z_values):
        if abs(z_values[scan_idx] - current_layer_z) <= z_tol:
            in_layer_indices.append(scan_idx)
            current_layer_z = float(np.median(z_values[in_layer_indices]))
            scan_idx += 1
            continue

        same_layer_ahead, _ = is_stable_window(z_values, scan_idx + 1, min_nodes, z_tol)
        if same_layer_ahead:
            next_window = z_values[scan_idx + 1:scan_idx + 1 + min_nodes]
            if np.all(np.abs(next_window - current_layer_z) <= z_tol):
                scan_idx += 1
                continue

        next_layer_idx = scan_idx + 1
        found_next_layer = False
        while next_layer_idx < len(z_values):
            next_stable, next_layer_z = is_stable_window(z_values, next_layer_idx, min_nodes, z_tol)
            if not next_stable:
                next_layer_idx += 1
                continue

            layer_step = next_layer_z - current_layer_z
            same_layer = abs(layer_step) <= z_tol
            expected_step = abs(layer_step - expected_layer_height) <= layer_height_tol
            if same_layer or expected_step:
                found_next_layer = True
                break

            next_layer_idx += 1

        if found_next_layer:
            break

        scan_idx += 1

    if len(in_layer_indices) >= min_nodes:
        layer_ids[in_layer_indices] = layer_id
        layer_z_values[in_layer_indices] = current_layer_z
        previous_layer_z = current_layer_z
        layer_id += 1

    current_idx = scan_idx + 1

df["z_bin"] = layer_z_values
df["layer"] = layer_ids

# Drop rows not belonging to stable extrusion layers
total_rows = len(df)
dropped_rows = int((df["layer"] < 0).sum())
dropped_pct = (dropped_rows / total_rows * 100) if total_rows else 0.0
print(f"Dropped rows not in stable extrusion layers: {dropped_rows} of {total_rows} ({dropped_pct:.2f}%)")
df = df[df["layer"] >= 0].copy()
df["layer"] = df["layer"].astype(int)
df["layer_median_z"] = df.groupby("layer")["Z"].transform("median")

base_x_match_tol = 0.08

def select_pass_nodes(layer_df, radius, max_y_by_pass, base_x_match_tol=0.08):
    unique_x = np.sort(layer_df["X"].dropna().unique())
    if len(unique_x) < 2 or layer_df["Y"].dropna().empty:
        raise ValueError("No X/Y coordinates found for the current layer")

    x_match_tol = base_x_match_tol
    if len(unique_x) > 1:
        positive_x_diffs = np.diff(unique_x)
        positive_x_diffs = positive_x_diffs[positive_x_diffs > 1e-6]
        if positive_x_diffs.size > 0:
            x_match_tol = max(x_match_tol, float(np.min(positive_x_diffs)) / 2)

    pass_x_values = np.sort(unique_x)[-2:]
    selected_nodes = []
    for pass_number, pass_x in ((1, pass_x_values[0]), (2, pass_x_values[1])):
        pass_candidates = layer_df[np.isclose(layer_df["X"], pass_x, atol=x_match_tol)].copy()
        if pass_candidates.empty:
            raise ValueError(
                "No pass {} node found near x={:.3f} for layer {}".format(
                    pass_number,
                    pass_x,
                    int(layer_df["layer"].iloc[0]),
                )
            )

        target_y = float(max_y_by_pass[pass_number] - pixel_to_mm * radius)
        pass_candidates["_target_y_distance"] = (pass_candidates["Y"] - target_y).abs()
        selected_nodes.append(
            pass_candidates.sort_values(["_target_y_distance", "Y"], ascending=[True, False]).iloc[0]
        )

    return selected_nodes

selected_node_rows = []
for layer, layer_df in df.groupby("layer", sort=True):
    first_pass_node, second_pass_node = select_pass_nodes(
        layer_df,
        radius=radius_value,
        max_y_by_pass=theoretical_max_y_by_pass,
        base_x_match_tol=base_x_match_tol,
    )

    for pass_number, node in ((1, first_pass_node), (2, second_pass_node)):
        selected_row = node[original_columns].to_dict()
        selected_row["layer_number"] = int(layer)
        selected_row["pass_number"] = int(pass_number)
        selected_node_rows.append(selected_row)

selected_node_df = (
    pd.DataFrame(selected_node_rows)
    .sort_values(["layer_number", "pass_number"])
    .reset_index(drop=True)
)

output_csv_file = os.path.join(os.path.dirname(csv_file), output_csv_name)
selected_node_df.to_csv(output_csv_file, index=False)

print("Detected layers:", df["layer"].nunique())
print("Detected radius:", radius_value)
print("Saved selected pass-node rows to:", output_csv_file)
print("Rows saved:", len(selected_node_df))

selected_node_df

df.head()


global_x_min = df["X"].min()
global_x_max = df["X"].max()
global_y_min = df["Y"].min()
global_y_max = df["Y"].max()

axis_padding_ratio = 0.05
x_padding = max((global_x_max - global_x_min) * axis_padding_ratio, 1e-9)
y_padding = max((global_y_max - global_y_min) * axis_padding_ratio, 1e-9)
x_limits = [global_x_min - x_padding, global_x_max + x_padding]
y_limits = [global_y_min - y_padding, global_y_max + y_padding]

temp_min = 60
temp_max = 200
pydeck_colorscale = [
    [0.0, "rgb(0, 0, 255)"],
    [0.5, "rgb(128, 255, 128)"],
    [1.0, "rgb(255, 0, 0)"]
]



Select Merged file
Dropped rows not in stable extrusion layers: 33 of 5318 (0.62%)
Detected layers: 34
Detected radius: 2.0
Saved selected pass-node rows to: C:/Users/seans/OneDrive - NTNU/Y5/master/work/code/automation_test/data/pa6cf/cal_21_high_gain/cal_20260415_100911_r2_to_8/R2\offset_pass_nodes.csv
Rows saved: 68


In [2]:
from IPython.display import display

layer_selector_2d_chrono = widgets.IntSlider(
    value=int(df["layer"].min()),
    min=int(df["layer"].min()),
    max=int(df["layer"].max()),
    step=1,
    description="Layer"
)

chronological_toggle = widgets.Checkbox(
    value=False,
    description="Chronological"
)

node_slider = widgets.IntSlider(
    value=1,
    min=1,
    max=1,
    step=1,
    description="Node",
    disabled=True
)

node_player = widgets.Play(
    value=1,
    min=1,
    max=1,
    step=1,
    interval=80,
    description="Play",
    disabled=True
)

widgets.jslink((node_player, "value"), (node_slider, "value"))

def update_node_controls(*_):
    layer_df = df[df["layer"] == layer_selector_2d_chrono.value]
    max_nodes = max(len(layer_df), 1)
    node_slider.max = max_nodes
    node_player.max = max_nodes
    node_slider.value = min(max(node_slider.value, 1), max_nodes)
    node_player.value = min(max(node_player.value, 1), max_nodes)
    is_enabled = chronological_toggle.value
    node_slider.disabled = not is_enabled
    node_player.disabled = not is_enabled

layer_selector_2d_chrono.observe(update_node_controls, names="value")
chronological_toggle.observe(update_node_controls, names="value")
update_node_controls()

def plot_layer_2d_chronological(layer, chronological, node_index):
    layer_df = df[df["layer"] == layer].reset_index(drop=True)
    layer_median = layer_df["layer_median_z"].iloc[0]
    fig = go.Figure()

    if chronological:
        node_index = min(max(int(node_index), 1), len(layer_df))
        traversed_df = layer_df.iloc[:node_index]
        remaining_df = layer_df.iloc[node_index:]
        current_df = layer_df.iloc[[node_index - 1]]

        if not remaining_df.empty:
            fig.add_trace(go.Scatter(
                x=remaining_df["X"],
                y=remaining_df["Y"],
                mode="markers",
                marker=dict(size=6, color="rgba(160, 160, 160, 0.18)"),
                hoverinfo="skip",
                showlegend=False
            ))

        fig.add_trace(go.Scatter(
            x=traversed_df["X"],
            y=traversed_df["Y"],
            mode="lines",
            line=dict(color="rgba(0, 0, 0, 0.35)", width=2),
            hoverinfo="skip",
            showlegend=False
        ))

        fig.add_trace(go.Scatter(
            x=traversed_df["X"],
            y=traversed_df["Y"],
            mode="markers",
            marker=dict(
                size=8,
                color=traversed_df["p_temp"],
                colorscale=pydeck_colorscale,
                cmin=temp_min,
                cmax=temp_max,
                colorbar=dict(title="Temp"),
                opacity=0.8
            ),
            customdata=traversed_df[["elapsed_hms", "p_index", "Z"]],
            hovertemplate=(
                "<b>Time:</b> %{customdata[0]}<br>"
                "<b>Temp:</b> %{marker.color:.1f}<br>"
                "<b>Pixel:</b> %{customdata[1]}<br>"
                "<b>Position:</b> (%{x:.3f}, %{y:.3f})<br>"
                "<b>Real Z:</b> %{customdata[2]:.3f}"
                "<extra></extra>"
            ),
            showlegend=False
        ))

        fig.add_trace(go.Scatter(
            x=current_df["X"],
            y=current_df["Y"],
            mode="markers",
            marker=dict(size=14, color="cyan", line=dict(color="black", width=1)),
            customdata=current_df[["elapsed_hms", "p_temp", "p_index", "Z"]],
            hovertemplate=(
                "<b>Current node:</b> " + str(node_index) + "<br>"
                "<b>Time:</b> %{customdata[0]}<br>"
                "<b>Temp:</b> %{customdata[1]:.1f}<br>"
                "<b>Pixel:</b> %{customdata[2]}<br>"
                "<b>Position:</b> (%{x:.3f}, %{y:.3f})<br>"
                "<b>Real Z:</b> %{customdata[3]:.3f}"
                "<extra></extra>"
            ),
            showlegend=False
        ))

        title = f"Layer {layer}: {layer_median:.3f} | Node {node_index}/{len(layer_df)}"
    else:
        fig.add_trace(go.Scatter(
            x=layer_df["X"],
            y=layer_df["Y"],
            mode="markers",
            marker=dict(
                size=8,
                color=layer_df["p_temp"],
                colorscale=pydeck_colorscale,
                cmin=temp_min,
                cmax=temp_max,
                colorbar=dict(title="Temp"),
                opacity=0.8
            ),
            customdata=layer_df[["elapsed_hms", "p_index", "Z"]],
            hovertemplate=(
                "<b>Time:</b> %{customdata[0]}<br>"
                "<b>Temp:</b> %{marker.color:.1f}<br>"
                "<b>Pixel:</b> %{customdata[1]}<br>"
                "<b>Position:</b> (%{x:.3f}, %{y:.3f})<br>"
                "<b>Real Z:</b> %{customdata[2]:.3f}"
                "<extra></extra>"
            ),
            showlegend=False
        ))

        title = f"Layer {layer}: {layer_median:.3f}"

    fig.update_layout(
        title=title,
        xaxis_title="X",
        yaxis_title="Y",
        height=700
    )
    fig.update_xaxes(range=x_limits)
    fig.update_yaxes(range=y_limits, scaleanchor="x", scaleratio=1)
    fig.show()

controls = widgets.HBox([
    layer_selector_2d_chrono,
    chronological_toggle,
    node_player,
    node_slider
])

output = widgets.interactive_output(
    plot_layer_2d_chronological,
    {
        "layer": layer_selector_2d_chrono,
        "chronological": chronological_toggle,
        "node_index": node_slider
    }
)

display(controls, output)


Output()